In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pathlib
import datetime as dt

In [ ]:
# results_path = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Gals/results')
# results_path = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Kerzers/CHYN-Kerzers01/low-cost_sensor_results_optimized/test01')
results_path = pathlib.Path('/Users/alexnaokiasatokobayashi/Documents/data/Kerzers/CHYN-Kerzers01/low-cost_sensor_results_optimized/plot4')
files = list(results_path.glob('*bestPareto.nc'))
nc_list = []
for file in files:
    # if not file.name.startswith('test8k'):
    #     continue
    # else:
    #     pass
    print(file.name)
    nc_list.append(file)

In [ ]:
nc_list

In [ ]:
ds = None
for file in files:
    print(file.name)
    if ds is None:
        ds = xr.open_dataset(file)
        print(ds.time.min())
    else:
        ds2 = xr.open_dataset(file)
        ds = xr.concat([ds, ds2], dim='time')
        print(ds2.time.min())
    print()
    print()
    # print(ds)


In [ ]:
ds

In [ ]:
ds.time.values

In [ ]:
ds2 = ds.sortby('time')

In [ ]:
ds2.where(ds2['dcdt(HM)']>0)

In [ ]:
ds2.time

In [ ]:
fig, ax = plt.subplots(figsize=(12,6), dpi=300)

ds2 = ds2.where((ds2['dcdt(HM)']>0.1)&(ds2['dcdt(HM)']<0.9))

# filter dates: set start and end date here
start_date = pd.to_datetime('2025-09-25')
end_date = pd.to_datetime('2025-10-06')
ds2 = ds2.sel(time=slice(start_date, end_date))

dcdt_median = ds2['dcdt(HM)'].median(dim='MC').values
dcdt_q16 = ds2['dcdt(HM)'].quantile(0.16, dim='MC').values
dcdt_q84 = ds2['dcdt(HM)'].quantile(0.84, dim='MC').values

print(dcdt_q84-dcdt_q16)
time = ds2['time'].values

days = pd.to_datetime(time).normalize().unique()
print(days)

# print(ds['dcdt(HM)'].median(dim='MC').values)
# ax.scatter(time, dcdt_median, label='median', s=1)
ax.fill_between(time, dcdt_q16, dcdt_q84, color='gray', alpha=0.8, label='16-84th percentile')
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.grid()

import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.DayLocator())          # daily ticks
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # or '%m-%d', etc.
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')


In [ ]:
ds['sgf']

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

dcdt_median = ds['sgf'].median(dim='MC').values
time = ds['time'].values

# print(ds['dcdt(HM)'].median(dim='MC').values)
ax.scatter(time, dcdt_median, label='median', s=1)
ax.set_ylabel('Soil Gas Flux ($\mu mol\ m^{-2}\ s^{-1}$)')
ax.set_xlabel('Time')
# ax.set_title('Soil Gas Flux over Time')
# ax.
ax.grid(True)

In [ ]:
12*1e-6*24*60*60*1e4/1e3


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

dcdt_median = ds['sgf'].median(dim='MC').values*12*1e-6*24*60*60*1e4/1e3
time = ds['time'].values

# print(ds['dcdt(HM)'].median(dim='MC').values)
ax.scatter(time, dcdt_median, label='median', s=1)
ax.set_ylabel('Soil Gas Flux ($Kg\ C\ ha^{-1}\ day^{-1}$)')
ax.set_xlabel('Time')
# ax.set_title('Soil Gas Flux over Time')
# ax.
ax.grid(True)

In [ ]:
fig, ax = plt.subplots()
for file in files:
    ds = xr.open_dataset(file)
    # print(ds)
    
    # ds['dcdt(HM)'].median(dim='MC').plot(label=file.stem, ax=ax)
    #moving average
    ds['dcdt(HM)'].rolling(time=5, center=True).mean().median(dim='MC').plot(label=file.stem+'_MA', ax=ax)

    # q16 = ds['dcdt(HM)'].quantile(0.16, dim='MC')
    # q84 = ds['dcdt(HM)'].quantile(0.84, dim='MC')
    # ax.fill_between(ds['time'], q16, q84, alpha=0.3)

    ax.legend()
    # break

In [ ]:
fig, ax = plt.subplots()

for file in files:
    ds = xr.open_dataset(file)
    # print(ds)
    
    # ds['dcdt(HM)'].median(dim='MC').plot(label=file.stem, ax=ax)
    #moving average
    # ds['dcdt(HM)'].rolling(time=5, center=True).mean().median(dim='MC').plot(label=file.stem+'_MA', ax=ax)

    q16 = ds['dcdt(HM)'].quantile(0.16, dim='MC')
    q84 = ds['dcdt(HM)'].quantile(0.84, dim='MC')
    diff = q84 - q16
    diff.plot(label=file.stem+'_IQR', ax=ax)
    # ax.fill_between(ds['time'], q16, q84, alpha=0.3)

    ax.legend()
    # break